<a href="https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

**Provisional lane (confirmed for Week 4):** Refresh / Content Opportunity Scoring  
**Decision:** Which existing content pages should editors inspect first?  
**Source:** FlyRank's approved, pseudonymized **30,000-row starter dataset** (`data/raw/content_refresh_anonymized.csv`).

This is my **transparent baseline** for a subsequent model to beat. A review recommendation is **not** an automatic rewrite, a claim of stale factual content, or proof of future search decline.

**Why starter data?** The ML-07 assignment Q&A explicitly allows the starter CSV. It contains the staleness signal used by FlyRank refresh-flag logic. My ML-04 warehouse experiment used different March feature/outcome windows; **its scores cannot be compared directly with this snapshot baseline**. A later model must evaluate on the *same dataset, slice, outcome, and split* or rebuild the baseline on the warehouse.

**Skills consulted:** `skills/README.md` → `building-baselines/SKILL.md` and `flyrank/flyrank-data/SKILL.md`. The reference scripts stay unchanged; results are created only in `work/`.

**Core first, AI second:** I will interpret the tables and top-ten claims personally before submitting. No private client names, URLs, raw queries, Hugging Face tokens, or full data exports go into GitHub.

## 1. Two signal checks **before** encoding the rule

**Unit:** one pseudonymized content page (one row). The starter measurements cover a trailing 90-day snapshot; no future-outcome dates are available here.

I will check two hypotheses using the **current-window declining proxy** solely as an *audit outcome*, not as an input to my rule:

1. **Staleness (flag-linked):** pages not updated for longer tend to show a larger fraction marked `trend_direction == "down"`. This corresponds to the *staleness* idea behind FlyRank refresh flags.
2. **Visibility (quick-win-related):** pages with greater 90-day search impressions tend to show a larger declining fraction. Visibility can matter for *editorial impact* even if it is **not** a good predictor of decline.

**Verdict rubric (set in advance):** Compare two pre-specified groups. If both groups have at least 30 pages: a difference of ≥5 percentage points in the expected direction is **CONFIRMED**; ≤−5 pp is **OPPOSITE**; otherwise **MIXED** (weak/ambiguous). If one group has fewer than 30 observations the verdict is **MIXED** (insufficient). The word refers **only to this sample-level directional test**, never to causation or a universal SEO rule. The audit's proxy is rule-defined from *current* impressions trends, not an independently measured future outcome.

My proposed review rule will remain an **untrained, fixed-weight baseline**; audit outcomes cannot be copied into its inputs.

In [1]:
# Setup — reads the safe starter CSV from MY GitHub copy in a fresh Colab runtime.
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython.display import display

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

def find_repo():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / DATA_REL).is_file():
            return folder
    if "google.colab" in sys.modules:
        root = Path("/content/flyrank-ml-internship")
        if not (root / DATA_REL).is_file():
            if root.exists():
                raise FileNotFoundError("Existing /content/flyrank-ml-internship has no starter CSV; inspect it before continuing.")
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/Jhonas05/flyrank-ml-internship.git", str(root)], check=True)
        return root
    raise FileNotFoundError("Run this notebook in Google Colab or from your repository clone.")

repo_root = find_repo()
df = pd.read_csv(repo_root / DATA_REL)
assert df["content_id"].is_unique, "Expected one row per content page"
assert {"impressions_90d", "days_since_last_update", "content_age_days", "trend_direction"}.issubset(df.columns)

# Shared editorial eligibility condition from my ML-02 / ML-03 framing.
lane = df.loc[
    (df["impressions_90d"] >= 100) &
    (df["content_age_days"] >= 90)
].copy()
assert len(lane) >= 50

# Audit-only outcome: NEVER feed into scoring rule.
lane["audit_declining_proxy"] = lane["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print(f"Starter data: {len(df):,} content pages")
print(f"Eligible review slice (>=100 impressions / >=90 days of content age): {len(lane):,} pages")
print(f"Current-window declining-proxy prevalence: {lane['audit_declining_proxy'].mean():.1%}")
print("Note: descriptive snapshot analysis, NOT forecasting or causal validation.")

Starter data: 30,000 content pages
Eligible review slice (>=100 impressions / >=90 days of content age): 22,006 pages
Current-window declining-proxy prevalence: 59.8%
Note: descriptive snapshot analysis, NOT forecasting or causal validation.


In [2]:
# Signal 1 — STALENESS audit, bucket table with n.
lane["staleness_bucket"] = pd.cut(
    lane["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0-30d", "31-90d", "91-180d", "181+d"]
)
staleness_table = (
    lane.groupby("staleness_bucket", observed=False)
    .agg(n=("audit_declining_proxy", "size"),
         declining_fraction=("audit_declining_proxy", "mean"),
         median_impressions=("impressions_90d", "median"))
)
staleness_table["declining_pct"] = (staleness_table["declining_fraction"] * 100).round(1)
print("Signal 1 / Flag-linked: DAYS SINCE UPDATE (n shown)")
display(staleness_table[["n", "declining_pct", "median_impressions"]])

def directional_verdict(low_n, low_rate, high_n, high_rate):
    if low_n < 30 or high_n < 30 or pd.isna(low_rate) or pd.isna(high_rate):
        return "MIXED", "insufficient comparison-group n"
    delta = 100 * (high_rate - low_rate)  # percentage points
    if delta >= 5:
        return "CONFIRMED", f"older/higher group +{delta:.1f} percentage points"
    if delta <= -5:
        return "OPPOSITE", f"older/higher group {delta:.1f} percentage points"
    return "MIXED", f"difference {delta:+.1f} pp is below the 5-pp evidence threshold"

fresh = lane.loc[lane["days_since_last_update"] <= 30, "audit_declining_proxy"]
old = lane.loc[lane["days_since_last_update"] > 90, "audit_declining_proxy"]
stale_verdict, stale_reason = directional_verdict(
    len(fresh), fresh.mean(), len(old), old.mean()
)
print(f"Signal 1 verdict: {stale_verdict} — {stale_reason}")
print(f"Comparison: <=30 days (n={len(fresh):,}) vs >90 days (n={len(old):,}).")

Signal 1 / Flag-linked: DAYS SINCE UPDATE (n shown)


,n,declining_pct,median_impressions
staleness_bucket,,,
0-30d,13735,58.3,1450.0
31-90d,152,59.2,688.0
91-180d,8084,62.2,2286.0
181+d,35,74.3,429.0


Signal 1 verdict: MIXED — difference +4.0 pp is below the 5-pp evidence threshold
Comparison: <=30 days (n=13,735) vs >90 days (n=8,119).


In [3]:
# Signal 2 — VISIBILITY audit, bucket table with n.
lane["visibility_bucket"] = pd.cut(
    lane["impressions_90d"],
    bins=[99, 499, 2999, 29999, np.inf],
    labels=["100-499", "500-2,999", "3,000-29,999", "30,000+"]
)
visibility_table = (
    lane.groupby("visibility_bucket", observed=False)
    .agg(n=("audit_declining_proxy", "size"),
         declining_fraction=("audit_declining_proxy", "mean"),
         median_impressions=("impressions_90d", "median"))
)
visibility_table["declining_pct"] = (visibility_table["declining_fraction"] * 100).round(1)
print("Signal 2 / Quick-win-related: 90-DAY IMPRESSIONS (n shown)")
display(visibility_table[["n", "declining_pct", "median_impressions"]])

moderate = lane.loc[lane["impressions_90d"].between(100, 499), "audit_declining_proxy"]
high = lane.loc[lane["impressions_90d"] >= 3000, "audit_declining_proxy"]
volume_verdict, volume_reason = directional_verdict(
    len(moderate), moderate.mean(), len(high), high.mean()
)
print(f"Signal 2 verdict: {volume_verdict} — {volume_reason}")
print(f"Comparison: 100-499 (n={len(moderate):,}) vs >=3,000 (n={len(high):,}).")
print("Editorial relevance of visibility is separate from evidence of a higher declining rate.")

Signal 2 / Quick-win-related: 90-DAY IMPRESSIONS (n shown)


,n,declining_pct,median_impressions
visibility_bucket,,,
100-499,5280,60.4,251.5
"500-2,999",8443,62.1,1230.0
"3,000-29,999",7205,58.6,7249.0
"30,000+",1078,46.2,48675.0


Signal 2 verdict: MIXED — difference -3.4 pp is below the 5-pp evidence threshold
Comparison: 100-499 (n=5,280) vs >=3,000 (n=8,283).
Editorial relevance of visibility is separate from evidence of a higher declining rate.


### My interpretation of the two signals (read outputs, then edit this paragraph)

The tables above report **n**, percentages, and directional verdicts, not proof of causality. I will explain in my own words whether staleness and visibility were **CONFIRMED**, **OPPOSITE**, or **MIXED**, and whether I would still use each signal for *review prioritization*. In particular, higher impressions can imply higher *editorial opportunity* even when they do not predict decline. **I will not add the observed declining proxy to my score**.

*Before committing: replace this instructional paragraph with 2–4 sentences referring to the actual two computed verdicts and sample sizes.*

## 2. Encode ONE transparent baseline rule, reason code, action label and queue

**Plain-language rule:** Among pages with at least 100 impressions and at least 90 days of content age, prioritize a page based on how **long it has gone without an update** and **how visible it currently is**. Pages with higher baseline scores are reviewed first.

- `staleness_component = min(days_since_last_update / 180, 1)` (0–1)
- `visibility_component = log(1 + impressions_90d) / log(1 + maximum impressions across eligible pages)` (0–1)
- **ONE baseline score:** `100 × (0.55 × staleness_component + 0.45 × visibility_component)`.
- **ONE reason code:** `freshness_visibility_review` (means "review priority based on age since update and measured visibility," *not* "confirmed stale facts")
- **ONE action label:** `Editorial review for possible refresh`.

Weights **0.55/0.45 are fixed illustrative heuristics, not trained or chosen to maximize the observed labels**. `trend_direction`, `trend_pct`, the 30-day comparison-window columns, and model/product flags are *excluded* from the score. Content/client IDs are for local traceability only; they never become score inputs.

**Local/Colab output:** `work/outputs/baseline_action_score.csv`. It is **intentionally gitignored** and must not be committed. The notebook (with safe aggregated outputs) **is** the submission artifact.

In [4]:
# Section 2 — Build the queue. NO audit labels or future-window fields in scoring.
SCORE_INPUTS = ["days_since_last_update", "impressions_90d"]
BLOCKED_INPUTS = {
    "trend_direction", "trend_pct", "audit_declining_proxy",
    "impressions_last_30d", "impressions_prev_30d",
    "is_declining_label", "client_id", "content_id"
}
assert not (set(SCORE_INPUTS) & BLOCKED_INPUTS), "Leakage or identifiers in score!"

age = pd.to_numeric(lane["days_since_last_update"], errors="coerce").fillna(0).clip(lower=0)
visibility = pd.to_numeric(lane["impressions_90d"], errors="coerce").fillna(0).clip(lower=0)
max_visible = float(visibility.max())
assert max_visible >= 100
lane["staleness_component"] = (age / 180).clip(upper=1)
lane["visibility_component"] = np.log1p(visibility) / np.log1p(max_visible)
lane["baseline_score"] = (
    100 * (0.55 * lane["staleness_component"] + 0.45 * lane["visibility_component"])
).round(4)
lane["reason_code"] = "freshness_visibility_review"
lane["action_label"] = "Editorial review for possible refresh"

queue_fields = [
    "content_id",  # pseudonymous ID for the local CSV only, NOT displayed in GitHub outputs
    "baseline_score", "reason_code", "action_label",
    "impressions_90d", "days_since_last_update", "content_age_days",
    "staleness_component", "visibility_component"
]
queue = (
    lane[queue_fields].sort_values(
        ["baseline_score", "impressions_90d", "days_since_last_update"],
        ascending=[False, False, False], kind="stable"
    ).reset_index(drop=True)
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
assert queue["rank"].is_unique and queue["baseline_score"].notna().all()
assert set(queue["reason_code"]) == {"freshness_visibility_review"}
assert set(queue["action_label"]) == {"Editorial review for possible refresh"}

out_path = repo_root / "work/outputs/baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Ranked review queue: {len(queue):,} pages")
print("CSV written inside the runtime's repository clone:", out_path.relative_to(repo_root))
print("CSV is NOT a GitHub deliverable — rerun this notebook to regenerate it.")
display(queue.drop(columns="content_id").head(5).reset_index(drop=True))

Ranked review queue: 22,006 pages
CSV written inside the runtime's repository clone: work/outputs/baseline_action_score.csv
CSV is NOT a GitHub deliverable — rerun this notebook to regenerate it.


,rank,baseline_score,reason_code,action_label,impressions_90d,days_since_last_update,content_age_days,staleness_component,visibility_component
0,1,92.7236,freshness_visibility_review,Editorial review for possible refresh,61678,194,231,1.0,0.838303
1,2,92.5990,freshness_visibility_review,Editorial review for possible refresh,59472,194,231,1.0,0.835534
2,3,89.7315,freshness_visibility_review,Editorial review for possible refresh,25715,194,231,1.0,0.771812
3,4,87.4764,freshness_visibility_review,Editorial review for possible refresh,13299,193,231,1.0,0.721699
4,5,85.6570,freshness_visibility_review,Editorial review for possible refresh,7812,194,231,1.0,0.681266


In [5]:
# Optional descriptor on same-snapshot proxy (NOT holdout accuracy / future prediction).
# The proxy is used only AFTER rankings were fixed, for a descriptive check.
# Correct alignment by pseudonymous key in memory, never included in public display:
proxy_by_id = lane.set_index("content_id")["audit_declining_proxy"]
ranked_proxy = queue["content_id"].map(proxy_by_id).to_numpy(dtype=int)
current_p10 = float(ranked_proxy[:10].mean())
current_p50 = float(ranked_proxy[:50].mean())
base_rate = float(lane["audit_declining_proxy"].mean())
print(f"Current-window proxy base rate: {base_rate:.3f}")
print(f"Descriptive baseline P@10:       {current_p10:.3f}")
print(f"Descriptive baseline P@50:       {current_p50:.3f}")
print("CAUTION: same-snapshot proxy metrics; neither held-out nor time-forward validation.")
print("Compare future models only on SAME candidate slice, labels and evaluation split.")

Current-window proxy base rate: 0.598
Descriptive baseline P@10:       1.000
Descriptive baseline P@50:       0.640
CAUTION: same-snapshot proxy metrics; neither held-out nor time-forward validation.
Compare future models only on SAME candidate slice, labels and evaluation split.


## 3. Top-10 skeptical review — one line for each recommendation

The code prints **ten distinct, data-grounded, conditional review notes**: one action, actual signals and reason code, plus a reason why that recommendation **could be wrong**. These are *provisional hypotheses*, **not proof that a particular page needs an edit**.

Read each row, challenge it, and revise the review wording where needed. A successful baseline produces **human review tasks**, not automatic edits. No private client IDs or content URLs are printed.

**Review confidence:** provisional / requires human inspection for every row. The dataset does not include factual accuracy, editorial context, or seasonality explanations.

In [6]:
# Section 3 — Print exactly ten skeptical lines with action / why / wrong-if.
top10 = queue.head(10).copy()
assert len(top10) == 10

review_lines = []
for row in top10.itertuples(index=False):
    days = float(row.days_since_last_update)
    imp = int(row.impressions_90d)
    if days >= 180:
        possible_error = "evergreen material may still be accurate despite age"
    elif days >= 90:
        possible_error = "content may still satisfy the current search intent"
    else:
        possible_error = "recent updates may already address the real issue"
    if imp >= 30000:
        possible_error += "; many impressions alone do not show decline"
    reason = (f"{imp:,} 90d impressions; {days:.0f} days since update; "
              f"score={row.baseline_score:.1f}; code={row.reason_code}")
    line = (f"#{int(row.rank):02d} ACTION: {row.action_label} | "
            f"WHY: {reason} | WRONG IF: {possible_error}.")
    review_lines.append(line)

print("TOP TEN — provisional editor review, one line each")
for line in review_lines:
    print(line)
assert len(review_lines) == 10
print("\nPersonal review: after reading these, identify weak recommendations in Section 4.")

TOP TEN — provisional editor review, one line each
#01 ACTION: Editorial review for possible refresh | WHY: 61,678 90d impressions; 194 days since update; score=92.7; code=freshness_visibility_review | WRONG IF: evergreen material may still be accurate despite age; many impressions alone do not show decline.
#02 ACTION: Editorial review for possible refresh | WHY: 59,472 90d impressions; 194 days since update; score=92.6; code=freshness_visibility_review | WRONG IF: evergreen material may still be accurate despite age; many impressions alone do not show decline.
#03 ACTION: Editorial review for possible refresh | WHY: 25,715 90d impressions; 194 days since update; score=89.7; code=freshness_visibility_review | WRONG IF: evergreen material may still be accurate despite age.
#04 ACTION: Editorial review for possible refresh | WHY: 13,299 90d impressions; 193 days since update; score=87.5; code=freshness_visibility_review | WRONG IF: evergreen material may still be accurate despite age.
#

## 4. Weak picks + strict leakage check

**My skeptical questions:** Would I waste time reviewing an evergreen page? Could seasonal search demand explain high impressions? Does a recent update already address the presumed need? Does the score simply favor large pages without detecting a content problem?

Use the audit below to find **at least one potential weak pick**. The current-window `trend_direction` proxy is shown *only as a post-ranking diagnostic*, not used to compute scores or choose which pages are first. A non-declining page **may** still need a refresh; conversely a declining page does not prove the editor should rewrite it.

**Known limitation:** The starter CSV is one 90-day snapshot. Its present-period trend proxy is not an independently observed **future** outcome. To compare a Week-5 model fairly, we need the same candidates and outcome, plus a client-holdout and time-forward analysis when historical windows permit. ML-04's March warehouse numbers are NOT comparable as-is.

In [7]:
# Section 4 — Identify a skeptical example WITHOUT feeding proxy into score.
# Audit outcome is mapped AFTER ranking only; this never reorders candidates.
preview = queue.head(10).drop(columns=["content_id"]).copy()
preview["currently_declining_proxy"] = ranked_proxy[:10]
print("Top-10 post-hoc audit (no IDs, no labels used to rank):")
display(preview[["rank", "baseline_score", "days_since_last_update",
                 "impressions_90d", "currently_declining_proxy"]])

non_declining = preview[preview["currently_declining_proxy"] == 0]
if len(non_declining):
    weak = non_declining.iloc[0]
    print(f"Potential weak pick: rank #{int(weak['rank'])} — high score but NOT declining by the current proxy.")
    print("This does NOT prove the review is wrong; check actual content need and editor time cost.")
else:
    weak = preview.iloc[-1]
    print(f"Potential weak pick to challenge: rank #{int(weak['rank'])} — even a proxy-declining page may be evergreen or seasonal.")
    print("No observational trend proves the editorial action will improve results.")

assert len(SCORE_INPUTS) == 2 and not set(SCORE_INPUTS) & BLOCKED_INPUTS
assert all(c not in SCORE_INPUTS for c in ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"])
assert queue["reason_code"].nunique() == 1
assert "audit_declining_proxy" not in queue.columns
print("Leakage audit PASS: score uses only update age and observed 90-day visibility.")
print("Dataset / queue CSV remain local to Colab and gitignored; only notebook outputs are saved.")

Top-10 post-hoc audit (no IDs, no labels used to rank):


,rank,baseline_score,days_since_last_update,impressions_90d,currently_declining_proxy
0,1,92.7236,194,61678,1
1,2,92.5990,194,59472,1
2,3,89.7315,194,25715,1
3,4,87.4764,193,13299,1
4,5,85.6570,194,7812,1
5,6,85.5439,193,7558,1
6,7,83.8385,194,4590,1
7,8,83.8131,194,4556,1
8,9,83.7164,194,4429,1
9,10,80.4366,193,1697,1


Potential weak pick to challenge: rank #10 — even a proxy-declining page may be evergreen or seasonal.
No observational trend proves the editorial action will improve results.
Leakage audit PASS: score uses only update age and observed 90-day visibility.
Dataset / queue CSV remain local to Colab and gitignored; only notebook outputs are saved.


## 5. Self-check (personally review and tick after executing)

- [ ] I independently read the **two bucket tables**, noted n for every bucket, and can explain the printed one-word verdicts (**CONFIRMED / OPPOSITE / MIXED / FALSE**) with evidence.
- [ ] At least one audited signal is tied to a real FlyRank flag (**staleness**); no causal or future-accuracy claim was made.
- [ ] I can state my **one untrained rule**, score weights, single reason code and editor-facing action.
- [ ] The notebook wrote `work/outputs/baseline_action_score.csv` inside the runtime clone; I understand this CSV is **gitignored** and should not be pushed.
- [ ] Ten top-ranked recommendations include an action, numeric **why**, and conditional **what would make it wrong**; I reviewed them with skepticism.
- [ ] I identified at least one potential weak recommendation and confirmed that **trend / target columns never enter the score**.
- [ ] I read, revised and personalized the interpretation and notes rather than treating AI-generated reasoning as my own observations without review.
- [ ] **Runtime → Run all** passed; the executed notebook (not the CSV) is committed at `work/notebooks/w04_baseline_score.ipynb` in my public repository.

**Submission URL only:** `https://github.com/Jhonas05/flyrank-ml-internship`.

**Suggested commit:** `Complete ML-07: Baseline Action Score and Top-10 Review`.  
**Do not submit** until the two bucket verdicts, top-ten lines and executed outputs are visible on GitHub.